In [3]:

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

root = Path("../dataset/raw")

VIEWS = {
    "uit-loi-ra": root / "view1",
    "uit-loi-vao": root / "view2",
    "anh-mang": root / "view3",
    "green-parking": root / "view4",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def show_all_boxes(view_name, view_dir, batch_size=12):
    image_dir, label_dir = view_dir / "images", view_dir / "labels"

    if not image_dir.is_dir():
        print(f"Không tìm thấy thư mục ảnh: {image_dir.resolve()}")
        return
    if not label_dir.is_dir():
        print(f"Không tìm thấy thư mục nhãn: {label_dir.resolve()}")
        return

    paths = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    if not paths:
        print(f"Không có ảnh trong {image_dir.resolve()}")
        return

    missing_labels, empty_labels, invalid_labels = [], [], []
    cols = 3

    for start in range(0, len(paths), batch_size):
        batch = paths[start:start + batch_size]
        rows = (len(batch) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
        axes = list(axes.flat) if hasattr(axes, "flat") else [axes]

        for ax, img_path in zip(axes, batch):
            with Image.open(img_path) as im:
                image = im.convert("RGB")
            img_w, img_h = image.size

            ax.imshow(image)
            ax.set_title(img_path.name, fontsize=8)
            ax.axis("off")

            label_path = label_dir / f"{img_path.stem}.txt"
            if not label_path.exists():
                missing_labels.append(img_path.name)
                ax.set_title(f"THIẾU NHÃN: {img_path.name}", fontsize=8, color="red")
                continue

            lines = label_path.read_text(encoding="utf-8-sig").splitlines()
            valid_box_count = 0

            for line_no, line in enumerate(lines, start=1):
                if not line.strip():
                    continue

                parts = line.split()
                try:
                    if len(parts) != 5:
                        raise ValueError("Cần đúng 5 giá trị")
                    cls = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                    if cls < 0 or not all(map(lambda v: v == v and abs(v) != float("inf"), [x, y, w, h])):
                        raise ValueError("Giá trị không hợp lệ")
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                        raise ValueError("Tọa độ/kích thước ngoài phạm vi YOLO")
                    x1, y1 = (x - w / 2) * img_w, (y - h / 2) * img_h
                    x2, y2 = (x + w / 2) * img_w, (y + h / 2) * img_h
                    if x1 < 0 or y1 < 0 or x2 > img_w or y2 > img_h:
                        raise ValueError("Box vượt ra ngoài ảnh")

                    ax.add_patch(patches.Rectangle((x1, y1), w * img_w, h * img_h, linewidth=2, edgecolor="lime", facecolor="none"))
                    ax.text(x1, max(0, y1 - 4), f"class {cls}", color="black", fontsize=8, bbox=dict(facecolor="lime", alpha=0.8, pad=2))
                    valid_box_count += 1

                except ValueError as e:
                    invalid_labels.append(f"{label_path.name}, dòng {line_no}: {e}")

            if not lines or not any(line.strip() for line in lines):
                empty_labels.append(img_path.name)
                ax.set_title(f"NHÃN RỖNG: {img_path.name}", fontsize=8, color="orange")
            elif valid_box_count == 0:
                ax.set_title(f"KHÔNG CÓ BOX HỢP LỆ: {img_path.name}", fontsize=8, color="red")

        for ax in axes[len(batch):]:
            ax.axis("off")

        fig.suptitle(f"{view_name} | Ảnh {start + 1}–{start + len(batch)} / {len(paths)}", fontsize=14)
        plt.tight_layout()
        plt.show()

    print(f"\n===== Báo cáo {view_name} =====")
    print(f"Tổng số ảnh: {len(paths)}")
    print(f"Thiếu file nhãn: {len(missing_labels)}")
    print(f"File nhãn rỗng: {len(empty_labels)}")
    print(f"Lỗi dòng nhãn: {len(invalid_labels)}")

    if missing_labels:
        print("Ảnh thiếu nhãn (tối đa 20):", missing_labels[:20])
    if empty_labels:
        print("Ảnh có nhãn rỗng (tối đa 20):", empty_labels[:20])
    if invalid_labels:
        print("Dòng nhãn lỗi (tối đa 20):")
        for item in invalid_labels[:20]:
            print(" -", item)


In [ ]:
# show_all_boxes("uit-loi-ra", VIEWS["uit-loi-ra"], batch_size=12)
# show_all_boxes("uit-loi-vao", VIEWS["uit-loi-vao"], batch_size=12)
# show_all_boxes("anh-mang", VIEWS["anh-mang"], batch_size=12)
show_all_boxes("green-parking", VIEWS["green-parking"], batch_size=12)